In [1]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint
from utils.train import train_model
from utils.models import EfficientNetApi

In [2]:
seed = 42
shuffle = True
batch_size = 3
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'

data_dir = '../../../..'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [3]:
load_model = efficientnet_b0(
     weights=EfficientNet_B0_Weights.DEFAULT
)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

In [4]:
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

Using device: cuda


In [5]:
from sklearn.model_selection import train_test_split

df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train, df_val = train_test_split(df_train_, test_size=0.2, random_state=seed)
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

In [6]:
def remove_nonexistent_images(df, images_dir):
    """
    Remove rows from df where the image does not exist in images_dir.
    """
    image_ids = df['image_id'].apply(lambda x: os.path.join(images_dir, f"{x}.png"))
    existent_images = [os.path.isfile(path) for path in image_ids]
    df = df[existent_images]
    return df

df_train = remove_nonexistent_images(df_train, images_dir)
df_val = remove_nonexistent_images(df_val, images_dir)
df_test = remove_nonexistent_images(df_test, images_dir)

#### view data

In [7]:
(df_train.shape, df_val.shape, df_test.shape)

((7215, 5), (1805, 5), (1590, 4))

In [8]:
transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

In [9]:
df_train.columns = df_train.columns.str.strip()

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms, format="png")
valid_dataset = PandasDataset(images_dir, df_val, transforms=None, format="png")
test_dataset = PandasDataset(images_dir, df_test, transforms=None, format="png")

In [10]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(test_dataset)
)

In [23]:
optimizer = optim.Adam(model.parameters(), lr = init_lr / warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier = warmup_factor, total_epoch = warmup_epochs, after_scheduler=scheduler_cosine)

In [24]:
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/b0-png.txt",
    path_to_save_model="models/b0-png.pth",
    patience=5,
)

Epoch 1/50



100%|██████████| 602/602 [01:44<00:00,  5.76it/s]


VAL_LOSS     0.308
VAL_ACC      Mean: 49.430 | Std: 1.176 | 95% CI: [47.535, 51.413]
VAL_KAPPA    Mean: 0.791 | Std: 0.011 | 95% CI: [0.772, 0.809]
VAL_F1       Mean: 0.460 | Std: 0.012 | 95% CI: [0.441, 0.481]
VAL_RECALL   Mean: 0.458 | Std: 0.012 | 95% CI: [0.438, 0.478]
VAL_PRECISION Mean: 0.477 | Std: 0.012 | 95% CI: [0.456, 0.497]
Salvando o melhor modelo... 0.0 -> 0.7907400965773304
Epoch 2/50



100%|██████████| 602/602 [01:48<00:00,  5.54it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.320
VAL_ACC      Mean: 53.063 | Std: 1.193 | 95% CI: [51.080, 55.014]
VAL_KAPPA    Mean: 0.798 | Std: 0.012 | 95% CI: [0.778, 0.816]
VAL_F1       Mean: 0.475 | Std: 0.012 | 95% CI: [0.456, 0.496]
VAL_RECALL   Mean: 0.478 | Std: 0.012 | 95% CI: [0.458, 0.498]
VAL_PRECISION Mean: 0.478 | Std: 0.012 | 95% CI: [0.458, 0.499]
Salvando o melhor modelo... 0.7907400965773304 -> 0.7975856148916532
Epoch 3/50



...

VAL_LOSS     0.533
VAL_ACC      Mean: 60.896 | Std: 1.154 | 95% CI: [58.947, 62.825]
VAL_KAPPA    Mean: 0.811 | Std: 0.012 | 95% CI: [0.790, 0.830]
VAL_F1       Mean: 0.552 | Std: 0.012 | 95% CI: [0.532, 0.573]
VAL_RECALL   Mean: 0.549 | Std: 0.012 | 95% CI: [0.529, 0.569]
VAL_PRECISION Mean: 0.560 | Std: 0.013 | 95% CI: [0.539, 0.581]
Epoch 15/50



100%|██████████| 602/602 [01:35<00:00,  6.29it/s]


VAL_LOSS     0.517
VAL_ACC      Mean: 62.753 | Std: 1.150 | 95% CI: [60.776, 64.601]
VAL_KAPPA    Mean: 0.818 | Std: 0.012 | 95% CI: [0.797, 0.837]
VAL_F1       Mean: 0.564 | Std: 0.012 | 95% CI: [0.543, 0.584]
VAL_RECALL   Mean: 0.560 | Std: 0.012 | 95% CI: [0.539, 0.579]
VAL_PRECISION Mean: 0.575 | Std: 0.013 | 95% CI: [0.552, 0.595]
Epoch 16/50



100%|██████████| 602/602 [01:36<00:00,  6.24it/s]


VAL_LOSS     0.592
VAL_ACC      Mean: 61.966 | Std: 1.145 | 95% CI: [60.111, 63.878]
VAL_KAPPA    Mean: 0.810 | Std: 0.013 | 95% CI: [0.789, 0.831]
VAL_F1       Mean: 0.568 | Std: 0.012 | 95% CI: [0.549, 0.589]
VAL_RECALL   Mean: 0.568 | Std: 0.012 | 95% CI: [0.548, 0.588]
VAL_PRECISION Mean: 0.583 | Std: 0.012 | 95% CI: [0.562, 0.603]

Early stopping at epoch 16. No improvement for 5 epochs.
Best epoch: 11 with kappa: 0.8189


# tests

In [11]:
from utils.metrics import evaluation, format_metrics
model.load_state_dict(
    torch.load(f"models/b0-png.pth")
)
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print(result)

100%|██████████| 530/530 [01:15<00:00,  7.01it/s]


VAL_ACC      Mean: 58.625 | Std: 1.239 | 95% CI: [56.604, 60.692]
VAL_KAPPA    Mean: 0.812 | Std: 0.012 | 95% CI: [0.791, 0.831]
VAL_F1       Mean: 0.529 | Std: 0.013 | 95% CI: [0.508, 0.550]
VAL_RECALL   Mean: 0.527 | Std: 0.013 | 95% CI: [0.507, 0.549]
VAL_PRECISION Mean: 0.543 | Std: 0.013 | 95% CI: [0.521, 0.564]


In [12]:
confusion_matrix = response[0].get("confusion_matrix")
print(confusion_matrix)

[[0.81987767 0.15004849 0.01606332 0.00222566 0.00945588 0.00232898]
 [0.12020871 0.6311438  0.21625616 0.02990623 0.0024851  0.        ]
 [0.03044371 0.27209729 0.4322345  0.1703245  0.07459236 0.02030764]
 [0.0269178  0.09790423 0.18272968 0.2937436  0.33958007 0.05912462]
 [0.05310724 0.10711067 0.04202598 0.11790581 0.53152767 0.14832261]
 [0.02709418 0.03264403 0.04270883 0.11400753 0.32718051 0.45636493]]
